# Data audit

**Objective.** Validate the analytic form. Checks: duplicates, missing values, item codes, score consistency, timing anomalies and the extremes of total time.

**Data.** Real mode needs the PISA 2025 public-use files processed as described in `data/README.md` (licensed by the OECD and never committed). For Colab, upload `data/processed/math_form*.parquet/.json` and `student_external.parquet` to the same relative paths, or mount Google Drive. Demo mode generates synthetic data. **Demo outputs are not findings.**

Errors are raised, never swallowed. Every output is written to `results/` with the form name; outputs from synthetic data are prefixed `demo_`.

In [ ]:
# --- Environment (Colab or local) -------------------------------------------
# Set REPO_URL to the GitHub repository once it exists (it is NOT public yet; no URL is assumed here).
REPO_URL = ""          # e.g. "https://github.com/<owner>/ability-adjusted-mcq-profiling.git"
DEMO = True            # True = synthetic demonstration (no PISA data needed); False = real PISA analysis
import os, sys, subprocess, platform
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not REPO_URL:
        raise SystemExit("Set REPO_URL (repository not yet published) or upload the repository folder to /content.")
    if not os.path.exists("ability-adjusted-mcq-profiling"):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("ability-adjusted-mcq-profiling")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.exists("configs") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")
sys.path.insert(0, "src"); sys.path.insert(0, "scripts")
import numpy, scipy, pandas, sklearn
print(platform.python_version(), "numpy", numpy.__version__, "scipy", scipy.__version__,
      "pandas", pandas.__version__, "sklearn", sklearn.__version__)
FORM = "demo" if DEMO else "math_formA"
print("Mode:", "SYNTHETIC DEMONSTRATION - not empirical results" if DEMO else "REAL PISA 2025 data")

In [ ]:
import pandas as pd, numpy as np, json
from run_experiments import load_form
df, meta = load_form(FORM, DEMO)
items = [c for c in df.columns if c.startswith("I") and c[1:].isdigit()]
print("rows", len(df), "items", len(items), "fingerprint", meta.get("fingerprint"))
print("duplicated ids:", df.duplicated(["CNT", "CNTSTUID"]).sum())
print("item codes:", pd.unique(df[items].values.ravel()))
print("score consistency:", np.allclose(df[items].sum(1, skipna=True), df.final_score))
print(df.total_duration_minutes.describe(percentiles=[.01,.05,.5,.95,.99]))
print("missing total time:", df.total_duration_minutes.isna().sum(), " >60 min:", (df.total_duration_minutes > 60).sum())